# Table 3.   Factors associated with persisting untreated Pain, Multivariate Analysis

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/data/rega_data/trauma_categories_Rega Pain Study15.09.2025_v2.xlsx'
medic_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/data/rega_data/rega_physician_list_09102025.xlsx'
meta_medic_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/data/medreg_extraction/joined_final_complete_extractions_20251008_221735.xlsx'

In [ ]:
restrict_to_trauma = True
restrict_to_primary = True

In [ ]:
# Load and preprocess data
data_df = pd.read_excel(data_path)
medic_df = pd.read_excel(medic_data_path)
meta_medic_df = pd.read_excel(meta_medic_data_path)

# Merge with physician data
medic_df['full_name'] = medic_df['Mitglieder mit Einsatzfunktion'].str.replace(' (Flugarzt/Flugärztin)', '')
medic_df.drop_duplicates(subset=['Mitglieder mit Einsatzfunktion'], inplace=True)
medic_df = medic_df.merge(meta_medic_df, how='left', on='full_name')
medic_df.rename(columns={'Sex m/w': 'physician_sex'}, inplace=True)
data_df = data_df.merge(medic_df, how='left', left_on='Mitglieder mit Einsatzfunktion', right_on='Mitglieder mit Einsatzfunktion')

# Remove duplicates
data_df = data_df.drop_duplicates(subset=["SNZ Ereignis Nr. "])

# Filter patients with VAS > 3 at scene
data_df = data_df[data_df["VAS_on_scene"] > 3]

print(f"Total patients after filtering: {len(data_df)}")
print(f"Adult patients: {(data_df['Alter '] >= 16).sum()}")
print(f"Pediatric patients: {(data_df['Alter '] < 16).sum()}")

In [ ]:
if restrict_to_trauma:
    n_non_trauma = data_df[data_df['Einteilung (reduziert)'] != 'Unfall'].shape[0]
    print(f'Excluded {n_non_trauma} non-trauma patients')

    # adult non-trauma patients
    n_adult_non_trauma = data_df[(data_df['Einteilung (reduziert)'] != 'Unfall') & (data_df["Alter "] >= 16)].shape[0]
    print(f'Excluded {n_adult_non_trauma} adult non-trauma patients')
    # pediatric non-trauma patients
    n_pediatric_non_trauma = data_df[(data_df['Einteilung (reduziert)'] != 'Unfall') & (data_df["Alter "] < 16)].shape[0]
    print(f'Excluded {n_pediatric_non_trauma} pediatric non-trauma patients')

    data_df = data_df[data_df['Einteilung (reduziert)'] == 'Unfall']

In [ ]:
if restrict_to_primary:
    n_secondary = data_df[data_df['Einsatzart'] != 'Primär'].shape[0]
    print(f'Excluded {n_secondary} secondary transport patients')

    # adult secondary transport patients
    n_adult_secondary = data_df[(data_df['Einsatzart'] != 'Primär') & (data_df["Alter "] >= 16)].shape[0]
    print(f'Excluded {n_adult_secondary} adult secondary transport patients')
    # pediatric secondary transport patients
    n_pediatric_secondary = data_df[(data_df['Einsatzart'] != 'Primär') & (data_df["Alter "] < 16)].shape[0]
    print(f'Excluded {n_pediatric_secondary} pediatric secondary transport patients')
    data_df = data_df[data_df['Einsatzart'] == 'Primär']

In [ ]:
# Univariate logistic regression analysis
def univariate_logistic_regression(df, outcome_var, predictor_vars):
    """
    Perform univariate logistic regression for each predictor variable
    """
    results = []
    
    for var in predictor_vars:
        # Prepare data
        X = df[[var]].copy()
        y = df[outcome_var]
        
        # Add constant for intercept
        X_with_const = sm.add_constant(X)
        
        # Fit logistic regression
        try:
            model = sm.Logit(y, X_with_const).fit(disp=0)
            
            # Extract results
            coef = model.params[var]
            or_value = np.exp(coef)
            ci_lower = np.exp(model.conf_int().loc[var, 0])
            ci_upper = np.exp(model.conf_int().loc[var, 1])
            p_value = model.pvalues[var]
            
            results.append({
                'Variable': var,
                'Coefficient': coef,
                'OR': or_value,
                'CI_lower': ci_lower,
                'CI_upper': ci_upper,
                'P_value': p_value,
                'OR_CI': f"{or_value:.2f} ({ci_lower:.2f}-{ci_upper:.2f})",
                'P_formatted': f"{p_value:.3f}" if p_value >= 0.001 else "<0.001"
            })
            
        except Exception as e:
            print(f"Error with variable {var}: {e}")
            
    return pd.DataFrame(results)

In [ ]:
# Multivariate logistic regression analysis
def multivariate_logistic_regression(
    df,
    outcome_var,
    predictor_vars,
    normalize=False,
    interaction_terms=None,
    interaction_sep='*',
    drop_original_interaction_vars=False,
    ):
    """
    Perform multivariate logistic regression
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing the data
    outcome_var : str
        Name of the outcome variable
    predictor_vars : list
        List of predictor variable names
    normalize : bool, optional (default=False)
        If True, standardize all predictor variables (mean=0, std=1)
    interaction_terms : list, optional (default=None)
        List of interactions to add. Each element can be:
        - tuple/list of variable names, e.g. ('age', 'NACA')
        - string, e.g. 'age*NACA' (uses interaction_sep)
    interaction_sep : str, optional (default='*')
        Separator for parsing string interaction terms
    drop_original_interaction_vars : bool, optional (default=False)
        If True, drops the base variables used in interactions
    
    Returns:
    --------
    results_df : pandas.DataFrame
        DataFrame containing regression results
    model_stats : dict
        Dictionary containing model statistics
    model : statsmodels.discrete.discrete_model.BinaryResultsWrapper
        Fitted logistic regression model
    """
    # Prepare data
    X = df[predictor_vars].copy()
    y = df[outcome_var]
    
    # Add interaction terms if provided
    interaction_cols = []
    if interaction_terms:
        for term in interaction_terms:
            if isinstance(term, str):
                parts = [p.strip() for p in term.split(interaction_sep)]
            else:
                parts = [str(p).strip() for p in term]
            if len(parts) < 2:
                raise ValueError(f"Invalid interaction term: {term}")
            # ensure all parts exist
            missing = [p for p in parts if p not in X.columns]
            if missing:
                raise ValueError(f"Interaction term {term} has missing variables: {missing}")
            # build interaction column
            col_name = interaction_sep.join(parts)
            interaction_col = X[parts[0]].copy()
            for p in parts[1:]:
                interaction_col = interaction_col * X[p]
            X[col_name] = interaction_col
            interaction_cols.append(col_name)
        if drop_original_interaction_vars:
            drop_vars = sorted({p for term in interaction_terms for p in (term.split(interaction_sep) if isinstance(term, str) else term)})
            X = X.drop(columns=[v for v in drop_vars if v in X.columns])
    
    # Normalize predictor variables if requested
    if normalize:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        X = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)
    
    # Add constant for intercept
    X_with_const = sm.add_constant(X)
    
    # Fit logistic regression
    model = sm.Logit(y, X_with_const).fit(disp=0)
    
    # Extract results
    results = []
    for var in X.columns:
        coef = model.params[var]
        or_value = np.exp(coef)
        ci_lower = np.exp(model.conf_int().loc[var, 0])
        ci_upper = np.exp(model.conf_int().loc[var, 1])
        p_value = model.pvalues[var]
        
        results.append({
            'Variable': var,
            'Coefficient': coef,
            'OR': or_value,
            'CI_lower': ci_lower,
            'CI_upper': ci_upper,
            'P_value': p_value,
            'OR_CI': f"{or_value:.2f} ({ci_lower:.2f}-{ci_upper:.2f})",
            'P_formatted': f"{p_value:.3f}" if p_value >= 0.001 else "<0.001",
        })
    
    results_df = pd.DataFrame(results)
    
    # Add model statistics
    model_stats = {
        'Log-likelihood': model.llf,
        'AIC': model.aic,
        'BIC': model.bic,
        'Pseudo R-squared': model.prsquared,
        'LLR p-value': model.llr_pvalue
    }
    
    return results_df, model_stats, model

In [ ]:
def preprocess_body_region(df):
    # fill na in Körperregion with Körperregion2
    df['Körperregion'] = df['Körperregion'].fillna(df['Körperregion2'])
    # set to lower
    df['Körperregion'] = df['Körperregion'].str.lower()
    # replace kopf and 'schädel-hirn/hals' and 'schädel-hirn/gesicht' and 'schädel-hirn/hals' with schädel-hirn
    df['Körperregion'] = df['Körperregion'].str.replace('kopf', 'schädel-hirn')
    df['Körperregion'] = df['Körperregion'].str.replace('schädel-hirn/hals', 'schädel-hirn')
    df['Körperregion'] = df['Körperregion'].str.replace('schädel-hirn/gesicht', 'schädel-hirn')
    df['Körperregion'] = df['Körperregion'].str.replace('schädel-hirn/hals', 'schädel-hirn')
    # replace 'kopf/gesicht' / 'augen' / 'kopf/hals' / 'kopf (gehör)' / 'kopf/gesicht/hals'  and 'schädel-hirn (gehör)' with 'gesicht'
    # df['Körperregion'] = df['Körperregion'].str.replace({'kopf/gesicht': 'gesicht', 'augen': 'gesicht', 'kopf/hals': 'gesicht', 'kopf (gehör)': 'gesicht', 'kopf/gesicht/hals': 'gesicht'})
    df['Körperregion'] = df['Körperregion'].str.replace('kopf/gesicht', 'gesicht')
    df['Körperregion'] = df['Körperregion'].str.replace('kopf/hals', 'gesicht')
    df['Körperregion'] = df['Körperregion'].str.replace('kopf (gehör)', 'gesicht')
    df['Körperregion'] = df['Körperregion'].str.replace('kopf/gesicht/hals', 'gesicht')
    df['Körperregion'] = df['Körperregion'].str.replace('augen', 'gesicht')
    df['Körperregion'] = df['Körperregion'].str.replace('schädel-hirn (gehör)', 'gesicht')
    # replace 'rücken' with 'wirbelsäule'
    df['Körperregion'] = df['Körperregion'].str.replace('rücken', 'wirbelsäule')
    # replace 'rump' with 'thorax'
    df['Körperregion'] = df['Körperregion'].str.replace('rumpf', 'thorax')
    # replace  'obere extremiät' and 'obere extermität' with 'obere extremität'    
    df['Körperregion'] = df['Körperregion'].str.replace('obere extremiät', 'obere extremität')
    df['Körperregion'] = df['Körperregion'].str.replace('obere extermität', 'obere extremität')
    # replace 'untere extermität' with 'untere extremität'    
    df['Körperregion'] = df['Körperregion'].str.replace('untere extermität', 'untere extremität')
    # replace '' with pd.NA
    df['Körperregion'] = df['Körperregion'].replace('', pd.NA)

    return df

# Adult Patients (≥16 years)

In [ ]:
# Prepare adult-only dataset (≥16 years)
from utils.utils import _extract_venous_access_features


adult_df = data_df[data_df['Alter '] >= 16].copy()

# insufficient pain management (VAS on arrival > 3)
adult_df['insufficient_pain_mgmt'] = (adult_df['VAS_on_arrival'] > 3).astype(int)

# Create predictor variables for adult patients (exclude pediatric variable)
adult_df['age'] = adult_df['Alter ']
adult_df['male_patient'] = (adult_df['Geschlecht'] == 'Männlich').astype(int)
adult_df['male_physician'] = (adult_df['physician_sex'] == 'm').astype(int)

# get only year of event
adult_df['event_year'] = pd.to_datetime(adult_df['Ereignisdatum'], format='%d.%m.%Y').dt.year
adult_df['physician_age'] = adult_df['event_year'] - adult_df['year_of_birth']
# physician year of final exam (from licence_date which can be either d.m.Y or Y)
adult_df['physician_licence_year'] = adult_df['licence_date'].apply(lambda x: str(x).split('.')[-1] if '.' in str(x) else str(x))
adult_df['physician_experience_years'] = adult_df['event_year'] - pd.to_numeric(adult_df['physician_licence_year'], errors='coerce')

adult_df['physician_anesthesiologist'] = adult_df['specialist_qualifications'].str.contains('Anaesthesiology', na=False).astype(int)
adult_df['physician_intensivist'] = adult_df['specialist_qualifications'].str.contains('Intensive care medicine', na=False).astype(int)
adult_df['physician_internist'] = adult_df['specialist_qualifications'].str.contains('General Internal Medicine|General medical practitioner', na=False).astype(int)

adult_df['primary_mission'] = (adult_df['Einsatzart'] == 'Primär').astype(int)
adult_df['night_mission'] = (adult_df['Tag oder Nacht'] == 'Nacht').astype(int)
# separate into winter half-year (October to March) and summer half-year (April to September)
adult_df['winter_season'] = np.where(adult_df['Monat'].isin(['Oktober', 'November', 'Dezember', 'Januar', 'Februar', 'März']), 1, 0).astype(int)
adult_df['trauma'] = adult_df['Einteilung (reduziert)'].str.contains('Unfall', na=False).astype(int)
adult_df['winch_extraction'] = adult_df['Bergungen'].str.contains('Winde', na=False).astype(int)
adult_df['vas_scene'] = adult_df['VAS_on_scene']
adult_df['mission_duration'] = (
    pd.to_datetime(adult_df['Übergabezeit'], format='%d.%m.%Y %H:%M:%S') - 
    pd.to_datetime(adult_df['Erstbefund'], format='%d.%m.%Y %H:%M:%S')
).dt.total_seconds() / 60

adult_df = preprocess_body_region(adult_df)

# number of regions involved per patient
adult_df['n_regions_involved'] = adult_df['Körperregion'].str.count(',') + 1
# if Körperregion is polytrauma, set n_regions_involved to 2
adult_df.loc[adult_df['Körperregion'].str.contains('polytrauma', na=False), 'n_regions_involved'] = 2

# regional involvement
adult_df['extremities_involved'] = adult_df['Körperregion'].str.contains('obere extremität|untere extremität', na=False).astype(int)
adult_df['thorax_involved'] = adult_df['Körperregion'].str.contains('thorax', na=False).astype(int)
# abdomen or pelvis involvement
adult_df['abdomen_pelvis_involved'] = adult_df['Körperregion'].str.contains('abdomen|becken', na=False).astype(int)
adult_df['spine_involved'] = adult_df['Körperregion'].str.contains('wirbelsäule', na=False).astype(int)
# face/neck involvement
adult_df['face_neck_involved'] = adult_df['Körperregion'].str.contains('gesicht|hals', na=False).astype(int)
# tbi 
adult_df['tbi_involved'] = adult_df['Körperregion'].str.contains('schädel-hirn', na=False).astype(int)
# polytrauma
adult_df['suspected_polytrauma'] = adult_df['Körperregion'].str.contains('polytrauma', na=False).astype(int)

# decompose patient / doctor sex into combinations
# male / male
adult_df['dr_male_pt_male'] = adult_df['male_physician'] & adult_df['male_patient']
adult_df['dr_female_pt_female'] = ((adult_df['male_physician'] == 0) & (adult_df['male_patient'] == 0)).astype(int)
adult_df['dr_male_pt_female'] = (adult_df['male_physician'] & (adult_df['male_patient'] == 0)).astype(int)
adult_df['dr_female_pt_male'] = ((adult_df['male_physician'] == 0) & adult_df['male_patient']).astype(int)

# Create the same additional variables for adult patients
adult_df['short_scene_time'] = (adult_df['mission_duration'] < 30).astype(int)

venous_access_features = _extract_venous_access_features(adult_df["Zugänge"])
adult_df['venous_access_count'] = venous_access_features['venous_access_count']
adult_df['no_venous_access'] = (adult_df['venous_access_count'] == 0).astype(int)

# Create medication dose variables (matching Table 1 approach)
adult_df['fentanyl_dose'] = 0
adult_df['ketamine_dose'] = 0
adult_df['esketamine_dose'] = 0
adult_df['morphine_dose'] = 0
adult_df['Alle Medikamente'] = adult_df['Alle Medikamente'].str.replace(',', ';')  # replace commas with semicolons for consistency
for i, row in adult_df.iterrows():
    if pd.isna(row['Alle Medikamente']) or row['Alle Medikamente'] == 0:
        continue
    for analgetic in row['Alle Medikamente'].split(';'):
        if analgetic.strip() == '':
            continue
        # remove mcg or mg from dose
        if '7IE' in analgetic:
                print(f"Skipping dose with 7IE: {analgetic}")
                continue

        analgetic = analgetic.replace('mcg', '').replace('mg', '').strip()
        if 'Fentanyl' in analgetic and '/h' not in analgetic:
            dose = analgetic.split('Fentanyl')[-1].strip()
            adult_df.at[i, 'fentanyl_dose'] += float(dose) 
        elif 'Fentanyl' in analgetic and '/h' in analgetic:
            dose = analgetic.split('Fentanyl')[-1].strip().replace('/h', '')
            dose = float(dose) * adult_df.at[i, 'mission_duration']  
            adult_df.at[i, 'fentanyl_dose'] += float(dose)
        elif 'Ketamin' in analgetic or 'Ketamine' in analgetic:
            dose = analgetic.split('Ketamin')[-1].strip()
            adult_df.at[i, 'ketamine_dose'] += float(dose)
        elif 'Esketamin' in analgetic:
            dose = analgetic.split('Esketamin')[-1].strip()
            adult_df.at[i, 'esketamine_dose'] += float(dose)
        elif 'Morphin' in analgetic or 'Morphine' in analgetic:
            dose = analgetic.split('Morphin')[-1].strip()
            adult_df.at[i, 'morphine_dose'] += float(dose)

# fentanyl given
adult_df['fentanyl_given'] = adult_df['fentanyl_dose'] > 0
adult_df['morphine_given'] = adult_df['morphine_dose'] > 0
adult_df['ketamine_given'] = adult_df['ketamine_dose'] > 0
adult_df['esketamine_given'] = adult_df['esketamine_dose'] > 0

# Create combined medication variables
adult_df['any_opiate_dose'] = adult_df['morphine_dose'] + adult_df['fentanyl_dose']
adult_df['any_opiate_given'] = (adult_df['morphine_dose'] > 0) | (adult_df['fentanyl_dose'] > 0)
adult_df['any_ketamine_given'] = (adult_df['ketamine_dose'] > 0) | (adult_df['esketamine_dose'] > 0)

# combination therapy (opiate + ketamine) vs monotherapy
adult_df['opiate_ketamine_combination_therapy'] = (adult_df['any_opiate_given'] & adult_df['any_ketamine_given']).astype(int)

# Create no_analgesic variable: patients who receive no analgesic
# (no_analgesic corresponds to a dose of any_opiate and any_ketamine of 0)
adult_df['no_analgesic'] = ((adult_df['any_opiate_given'] == 0) & (adult_df['any_ketamine_given'] == 0)).astype(int)

# create outcome variable: insufficient pain management (VAS on arrival > 3) and no analgesic given
adult_df['persisting_untreated_pain'] = ((adult_df['insufficient_pain_mgmt'] == 1) & (adult_df['no_analgesic'] == 1)).astype(int)

# Variables for adult model (including the 3 required variables)
adult_model_vars = ['persisting_untreated_pain', 'age', 'NACA',
                    'male_patient', 'male_physician',
                    'dr_male_pt_male', 'dr_female_pt_female', 'dr_male_pt_female', 'dr_female_pt_male',
                                        'physician_age', 'physician_experience_years', 'physician_anesthesiologist', 'physician_intensivist', 'physician_internist',
                    'n_regions_involved', 'suspected_polytrauma', 'extremities_involved', 'thorax_involved', 'abdomen_pelvis_involved',
                    'spine_involved', 'face_neck_involved', 'tbi_involved',
                   'mission_duration', 'night_mission', 'no_venous_access',
                    'winch_extraction', 'vas_scene', 'winter_season']

# print number of na per variable in adult_model_vars
for var in adult_model_vars:
    n_na = adult_df[var].isna().sum()
    print(f"{var}: {n_na} NA values")

adult_df_clean = adult_df[adult_model_vars].dropna()

print(f"Adult patients included in model: {len(adult_df_clean)}")

In [ ]:
# Adult predictor variables (including the 3 required variables)
adult_predictor_vars = ['age', 
                        'physician_age', 'physician_anesthesiologist', 'physician_intensivist', 'physician_internist',
                          'n_regions_involved', 'suspected_polytrauma', 'extremities_involved', 'thorax_involved', 'abdomen_pelvis_involved',
                    'spine_involved', 'face_neck_involved', 'tbi_involved',
                        'mission_duration', 'NACA',
                        'night_mission', 'winter_season', 'no_venous_access',
                       'winch_extraction', 'vas_scene']

simple_sex_vars = ['male_patient', 'male_physician']
composed_sex_vars = ['dr_male_pt_male',  'dr_male_pt_female', 'dr_female_pt_male']

# Rebuild cleaned dataset so physician_experience_years is fully excluded from sample selection
adult_required_vars = ['persisting_untreated_pain'] + list(dict.fromkeys(adult_predictor_vars + simple_sex_vars + composed_sex_vars))
adult_df_clean = adult_df[adult_required_vars].dropna()
print(f"Adult patients included in model (without physician experience): {len(adult_df_clean)}")

# Univariate analysis for adults (include doctor/patient sex-combination terms)
adult_univariate_vars = list(dict.fromkeys(adult_predictor_vars + composed_sex_vars))
adult_univariate_results = univariate_logistic_regression(adult_df_clean, 'persisting_untreated_pain', adult_univariate_vars)

print("ADULT PATIENTS - Univariate Analysis:")
print("=" * 60)
for _, row in adult_univariate_results.iterrows():
    print(f"{row['Variable']}: OR = {row['OR_CI']}, p = {row['P_formatted']}")

# Multivariate analysis for adults 
# simple regression to test interactions between patient and physician sex
simple_adult_multivariate_vars = adult_predictor_vars + simple_sex_vars
composed_adult_multivariate_vars = adult_predictor_vars + composed_sex_vars

# simple multivariate analysis should include interaction between patient and physician sex
simple_adult_multivariate_results, simple_adult_model_stats, simple_adult_fitted_model = multivariate_logistic_regression(
    adult_df_clean, 'persisting_untreated_pain', simple_adult_multivariate_vars, 
    interaction_terms=[('male_patient', 'male_physician')])

adult_multivariate_vars = composed_adult_multivariate_vars
adult_multivariate_results, adult_model_stats, adult_fitted_model = multivariate_logistic_regression(
    adult_df_clean, 'persisting_untreated_pain', adult_multivariate_vars)

print("\nPersisting untreated pain: (VAS on arrival > 3 and no analgesic given)")
print(f"\nADULT PATIENTS - Simple Multivariate Analysis:")
print("=" * 60)
for _, row in simple_adult_multivariate_results.iterrows():
    print(f"{row['Variable']}: OR = {row['OR_CI']}, p = {row['P_formatted']}")

print(f"\nADULT PATIENTS - Multivariate Analysis:")
print("=" * 60)
for _, row in adult_multivariate_results.iterrows():
    print(f"{row['Variable']}: OR = {row['OR_CI']}, p = {row['P_formatted']}")

print(f"\nAdult Model Statistics:")
print(f"Pseudo R-squared: {adult_model_stats['Pseudo R-squared']:.3f}")
print(f"AIC: {adult_model_stats['AIC']:.2f}")
print(f"LLR p-value: {adult_model_stats['LLR p-value']:.3f}")

# Multivariate analysis with normalized predictors
adult_multivariate_results_normalized, adult_model_stats_normalized, adult_fitted_model_normalized = multivariate_logistic_regression(
    adult_df_clean, 'persisting_untreated_pain', adult_multivariate_vars, normalize=True)

print(f"\nADULT PATIENTS - Multivariate Analysis (Normalized):")
print("=" * 60)
for _, row in adult_multivariate_results_normalized.iterrows():
    print(f"{row['Variable']}: OR = {row['OR_CI']}, p = {row['P_formatted']}")

# Ensure physician cluster IDs are available in cleaned frame for mixed-effects grouping
if 'physician_cluster_id' not in adult_df_clean.columns:
    adult_df_clean['physician_cluster_id'] = (
        adult_df.loc[adult_df_clean.index, 'Mitglieder mit Einsatzfunktion']
        .fillna('UNKNOWN_PHYSICIAN')
        .astype(str)
        .str.strip()
    )

# Mixed-effects logistic model (random intercept by physician)
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

def fit_mixed_effects_logit(df, outcome_var, predictor_vars, group_var, label="Population", normalize=False):
    """
    Fit a mixed-effects logistic regression with random intercept per physician.
    Tries MAP (Laplace) first; falls back to variational Bayes if needed.
    """
    model_df = df.copy()

    if normalize:
        scaler = StandardScaler()
        model_df[predictor_vars] = scaler.fit_transform(model_df[predictor_vars])

    fixed_formula = outcome_var + " ~ " + " + ".join(predictor_vars)
    vc_formula = {"physician_re": f"0 + C({group_var})"}

    model = BinomialBayesMixedGLM.from_formula(fixed_formula, vc_formula, model_df)

    fit_method = "MAP"
    try:
        result = model.fit_map()
    except Exception as map_err:
        print(f"{label}: MAP fit failed ({map_err}). Falling back to VB fit.")
        result = model.fit_vb()
        fit_method = "VB"

    fixed_terms = ["Intercept"] + predictor_vars
    fe_mean = result.fe_mean
    fe_sd = result.fe_sd
    z_values = fe_mean / fe_sd
    p_values = 2 * stats.norm.sf(np.abs(z_values))

    mixed_fixed_df = pd.DataFrame({
        "Variable": fixed_terms,
        "Coefficient": fe_mean,
        "SE": fe_sd,
        "z": z_values,
        "P_value": p_values,
        "OR": np.exp(fe_mean),
        "CI_lower": np.exp(fe_mean - 1.96 * fe_sd),
        "CI_upper": np.exp(fe_mean + 1.96 * fe_sd),
    })
    mixed_fixed_df["OR_CI"] = mixed_fixed_df.apply(
        lambda r: f"{r['OR']:.2f} ({r['CI_lower']:.2f}-{r['CI_upper']:.2f})", axis=1
    )
    mixed_fixed_df["P_formatted"] = mixed_fixed_df["P_value"].apply(
        lambda p: f"{p:.3f}" if p >= 0.001 else "<0.001"
    )

    norm_label = " (normalized)" if normalize else ""
    print(f"\n{label} mixed-effects model{norm_label} (random intercept by physician)")
    print("=" * 70)
    print(f"Fit method: {fit_method}")
    print(f"Observations: {len(model_df)}")
    print(f"Physician clusters: {model_df[group_var].nunique()}")

    return result, mixed_fixed_df

adult_mixed_result, adult_mixed_fixed_df = fit_mixed_effects_logit(
    adult_df_clean,
    outcome_var='persisting_untreated_pain',
    predictor_vars=adult_multivariate_vars,
    group_var='physician_cluster_id',
    label='Adult'
)

adult_mixed_result_normalized, adult_mixed_fixed_df_normalized = fit_mixed_effects_logit(
    adult_df_clean,
    outcome_var='persisting_untreated_pain',
    predictor_vars=adult_multivariate_vars,
    group_var='physician_cluster_id',
    label='Adult',
    normalize=True,
)

adult_multivariate_results

In [ ]:
from scipy import stats

# Mixed-effects normalized Wald and post-hoc tests for doctor/patient sex categories
sex_vars = ['dr_male_pt_male', 'dr_male_pt_female', 'dr_female_pt_male']
fixed_terms = ['Intercept'] + adult_multivariate_vars

# Build fixed-effects coefficient and covariance objects from normalized mixed model
beta = pd.Series(adult_mixed_result_normalized.fe_mean, index=fixed_terms)
p_fixed = len(fixed_terms)
cov_all = np.asarray(adult_mixed_result_normalized.cov_params())
cov_fixed = cov_all[:p_fixed, :p_fixed]
cov_fixed_df = pd.DataFrame(cov_fixed, index=fixed_terms, columns=fixed_terms)

# 1. Global Wald test (all sex_vars jointly): H0: beta_sex = 0
b = beta.loc[sex_vars].values.reshape(-1, 1)
V = cov_fixed_df.loc[sex_vars, sex_vars].values
W = float(b.T @ np.linalg.inv(V) @ b)
df_wald = len(sex_vars)
p_global = stats.chi2.sf(W, df_wald)

print("\nGlobal Wald test for doctor/patient sex categories (adult mixed-effects normalized model):")
print("=" * 95)
print(f"chi2({df_wald}) = {W:.3f}, p = {p_global:.4g}")

# 2. Pairwise post-hoc contrasts using normalized mixed-effects covariance
contrast_defs = {
    'dr_male_pt_male vs dr_male_pt_female': ('dr_male_pt_male', 'dr_male_pt_female'),
    'dr_male_pt_male vs dr_female_pt_male': ('dr_male_pt_male', 'dr_female_pt_male'),
    'dr_male_pt_female vs dr_female_pt_male': ('dr_male_pt_female', 'dr_female_pt_male'),
    'dr_male_pt_female vs dr_female_pt_female': ('dr_male_pt_female', None),
}

print("\nPairwise post-hoc contrasts for doctor/patient sex categories (adult mixed-effects normalized):")
print("=" * 95)

contrast_summary = []
for name, (a, b_ref) in contrast_defs.items():
    c = np.zeros(len(fixed_terms))
    c[fixed_terms.index(a)] = 1.0
    if b_ref is not None:
        c[fixed_terms.index(b_ref)] = -1.0

    est = float(c @ beta.values)
    se = float(np.sqrt(c @ cov_fixed @ c))
    z = est / se
    pval = 2 * stats.norm.sf(abs(z))

    or_val = np.exp(est)
    ci_low = np.exp(est - 1.96 * se)
    ci_high = np.exp(est + 1.96 * se)

    print(f"{name}: OR = {or_val:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {pval:.4f}")

    contrast_summary.append({
        'Contrast': name,
        'Logit_diff': est,
        'SE': se,
        'z_value': z,
        'OR': or_val,
        'CI_lower': ci_low,
        'CI_upper': ci_high,
        'p_value': pval,
    })

contrast_df = pd.DataFrame(contrast_summary)
contrast_df

In [ ]:
# Create formatted Table 3: Factors for Persisting Untreated Pain
def create_table3(
    univariate_df,
    multivariate_df,
    multivariate_mixed_df=None,
    multivariate_normalized_df=None,
    multivariate_mixed_normalized_df=None,
):
    """
    Create a publication-ready Table 3

    Parameters:
    -----------
    univariate_df : pandas.DataFrame
        Univariate analysis results
    multivariate_df : pandas.DataFrame
        Multivariate analysis results (non-normalized)
    multivariate_mixed_df : pandas.DataFrame, optional
        Mixed-effects logistic results (random intercept by physician)
    multivariate_normalized_df : pandas.DataFrame, optional
        Multivariate analysis results with normalized predictors
    multivariate_mixed_normalized_df : pandas.DataFrame, optional
        Mixed-effects logistic results with normalized predictors
    """
    # Define variable labels for better presentation
    variable_labels = {
        'age': 'Age (years)',
        'male_patient': 'Male patient',
        'male_physician': 'Male physician',
        'short_scene_time': 'Short time at scene (<30 min)',
        'no_analgesic': 'No analgesic given',
        'pediatric': 'Pediatric patient (<16 years)',
        'primary_mission': 'Primary mission',
        'night_mission': 'Night mission',
        'trauma': 'Trauma',
        'winch_extraction': 'Winch extraction',
        'vas_scene': 'NRS on scene',
        'mission_duration': 'Mission duration (minutes)',
        'dr_male_pt_male': 'Male doctor - Male patient',
        'dr_male_pt_female': 'Male doctor - Female patient',
        'dr_female_pt_male': 'Female doctor - Male patient',
        'n_regions_involved': 'Number of regions involved',
        'suspected_polytrauma': 'Suspected polytrauma',
        'extremities_involved': 'Extremities involved',
        'thorax_involved': 'Thorax involved',
        'abdomen_pelvis_involved': 'Abdomen/pelvis involved',
        'spine_involved': 'Spine involved',
        'face_neck_involved': 'Face/neck involved',
        'tbi_involved': 'Traumatic brain injury',
        'NACA': 'NACA score',
        'winter_season': 'Winter season'
    }

    # Collect variables available across all supplied model outputs
    all_vars = []
    seen = set()

    def _add_vars(df):
        if df is None or 'Variable' not in df.columns:
            return
        for v in df['Variable'].tolist():
            if v == 'Intercept' or v in seen:
                continue
            seen.add(v)
            all_vars.append(v)

    _add_vars(univariate_df)
    _add_vars(multivariate_df)
    _add_vars(multivariate_mixed_df)
    _add_vars(multivariate_normalized_df)
    _add_vars(multivariate_mixed_normalized_df)

    # Preferred row order to match reference table (physician_experience_years excluded)
    preferred_order = [
        'age',
        'dr_male_pt_male',
        'dr_male_pt_female',
        'dr_female_pt_male',
        'physician_age',
        'physician_anesthesiologist',
        'physician_intensivist',
        'physician_internist',
        'n_regions_involved',
        'suspected_polytrauma',
        'extremities_involved',
        'thorax_involved',
        'abdomen_pelvis_involved',
        'spine_involved',
        'face_neck_involved',
        'tbi_involved',
        'mission_duration',
        'NACA',
        'night_mission',
        'winter_season',
        'no_venous_access',
        'winch_extraction',
        'vas_scene',
    ]

    # Keep preferred order first, then append any remaining variables
    variable_order = [v for v in preferred_order if v in all_vars]
    variable_order += [v for v in all_vars if v not in variable_order]

    # Create combined table
    table3_data = []

    # Merge univariate and multivariate results
    for var in variable_order:
        # Check if variable is in univariate model
        uni_row = univariate_df[univariate_df['Variable'] == var]
        if len(uni_row) > 0:
            uni_row = uni_row.iloc[0]
            uni_or_ci = uni_row['OR_CI']
            uni_p = uni_row['P_formatted']
        else:
            uni_or_ci = '-'
            uni_p = '-'

        # Check if variable is in multivariate model
        multi_row = multivariate_df[multivariate_df['Variable'] == var]
        if len(multi_row) > 0:
            multi_row = multi_row.iloc[0]
            multi_or_ci = multi_row['OR_CI']
            multi_p = multi_row['P_formatted']
        else:
            multi_or_ci = '-'
            multi_p = '-'

        # Check if variable is in mixed-effects multivariate model
        if multivariate_mixed_df is not None:
            multi_mixed_row = multivariate_mixed_df[multivariate_mixed_df['Variable'] == var]
            if len(multi_mixed_row) > 0:
                multi_mixed_row = multi_mixed_row.iloc[0]
                multi_mixed_or_ci = multi_mixed_row['OR_CI']
                multi_mixed_p = multi_mixed_row['P_formatted']
            else:
                multi_mixed_or_ci = '-'
                multi_mixed_p = '-'
        else:
            multi_mixed_or_ci = None
            multi_mixed_p = None

        # Check if variable is in normalized multivariate model
        if multivariate_normalized_df is not None:
            multi_norm_row = multivariate_normalized_df[multivariate_normalized_df['Variable'] == var]
            if len(multi_norm_row) > 0:
                multi_norm_row = multi_norm_row.iloc[0]
                multi_norm_or_ci = multi_norm_row['OR_CI']
                multi_norm_p = multi_norm_row['P_formatted']
            else:
                multi_norm_or_ci = '-'
                multi_norm_p = '-'
        else:
            multi_norm_or_ci = None
            multi_norm_p = None

        # Check if variable is in normalized mixed-effects model
        if multivariate_mixed_normalized_df is not None:
            mixed_norm_row = multivariate_mixed_normalized_df[multivariate_mixed_normalized_df['Variable'] == var]
            if len(mixed_norm_row) > 0:
                mixed_norm_row = mixed_norm_row.iloc[0]
                mixed_norm_or_ci = mixed_norm_row['OR_CI']
                mixed_norm_p = mixed_norm_row['P_formatted']
            else:
                mixed_norm_or_ci = '-'
                mixed_norm_p = '-'
        else:
            mixed_norm_or_ci = None
            mixed_norm_p = None

        row_data = {
            'Variable': variable_labels.get(var, var),
            'Univariate_OR_CI': uni_or_ci,
            'Univariate_P': uni_p,
            'Multivariate_OR_CI': multi_or_ci,
            'Multivariate_P': multi_p
        }

        if multivariate_mixed_df is not None:
            row_data['Multivariate_Mixed_OR_CI'] = multi_mixed_or_ci
            row_data['Multivariate_Mixed_P'] = multi_mixed_p

        if multivariate_normalized_df is not None:
            row_data['Multivariate_Normalized_OR_CI'] = multi_norm_or_ci
            row_data['Multivariate_Normalized_P'] = multi_norm_p

        if multivariate_mixed_normalized_df is not None:
            row_data['Multivariate_Mixed_Normalized_OR_CI'] = mixed_norm_or_ci
            row_data['Multivariate_Mixed_Normalized_P'] = mixed_norm_p

        table3_data.append(row_data)

    table3_df = pd.DataFrame(table3_data)
    return table3_df

In [ ]:
# Create Table 3: Adult Patients
adult_table3 = create_table3(
    adult_univariate_results,
    adult_multivariate_results,
    multivariate_mixed_df=adult_mixed_fixed_df,
    multivariate_normalized_df=adult_multivariate_results_normalized,
    multivariate_mixed_normalized_df=adult_mixed_fixed_df_normalized,
)

print("Table 3. Factors for Persisting Untreated Pain - ADULT PATIENTS (≥16 years)")
print("=" * 160)
print(f"{'Variable':<30} {'Univariate':<20} {'P-value':<10} {'Multivariate':<20} {'P-value':<10} {'Mixed Effects':<20} {'P-value':<10} {'Mixed-Normalized':<20} {'P-value':<10}")
print(f"{'':30} {'OR (95% CI)':<20} {'':10} {'OR (95% CI)':<20} {'':10} {'OR (95% CI)':<20} {'':10} {'OR (95% CI)':<20} {'':10}")
print("-" * 160)

for _, row in adult_table3.iterrows():
    print(
        f"{row['Variable']:<30} "
        f"{row['Univariate_OR_CI']:<20} {row['Univariate_P']:<10} "
        f"{row['Multivariate_OR_CI']:<20} {row['Multivariate_P']:<10} "
        f"{row['Multivariate_Mixed_OR_CI']:<20} {row['Multivariate_Mixed_P']:<10} "
        f"{row['Multivariate_Mixed_Normalized_OR_CI']:<20} {row['Multivariate_Mixed_Normalized_P']:<10}"
    )

print("\nOR = Odds Ratio, CI = Confidence Interval")
print("Mixed Effects = Logistic mixed-effects model with random intercept by physician")
print("Mixed-Normalized OR represents change per 1 standard deviation increase in predictor")
print(f"Model includes {len(adult_df_clean)} adult patients")
print(f"Persisting untreated pain: {adult_df_clean['persisting_untreated_pain'].sum()}/{len(adult_df_clean)} ({adult_df_clean['persisting_untreated_pain'].mean():.1%})")

# Show significant variables for adults under normalized mixed-effects inference
adult_significant_vars_mixed_norm = adult_mixed_fixed_df_normalized[
    (adult_mixed_fixed_df_normalized['Variable'] != 'Intercept') & (adult_mixed_fixed_df_normalized['P_value'] < 0.05)
]
if len(adult_significant_vars_mixed_norm) > 0:
    print("\nSignificant Variables in Adult Normalized Mixed-Effects Model (p < 0.05):")
    for _, row in adult_significant_vars_mixed_norm.iterrows():
        direction = "increased" if row['OR'] > 1 else "decreased"
        print(f"- {row['Variable']}: OR = {row['OR']:.2f}, {direction} odds of persisting untreated pain")
else:
    print("\nNo variables reached statistical significance (p < 0.05) in adult normalized mixed-effects model")

adult_table3

In [ ]:
# save adult_table3 to csv
# adult_table3.to_csv('/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/adult_trauma_analgesia/analysis/adult_trauma/table3.csv', index=False)

In [ ]:
# For each evaluated factor, state the reference category for both univariate and multivariate analyses
from collections import OrderedDict

def _reference_category(var_name: str) -> str:
    # Continuous variables (no reference category; effect per unit increase)
    continuous_vars = {
        'age', 'physician_age', 'physician_experience_years', 'n_regions_involved',
        'mission_duration', 'NACA', 'vas_scene', 'venous_access_count'
    }
    if var_name in continuous_vars:
        return 'Continuous (per unit increase; no reference category)'

    # Doctor/patient sex combinations (reference is the omitted group)
    if var_name in {'dr_male_pt_male', 'dr_male_pt_female', 'dr_female_pt_male'}:
        return 'Female physician + Female patient (reference group)'

    # Binary indicators (0 = No/Absent)
    binary_reference = {
        'physician_anesthesiologist': 'Not an anesthesiologist (0)',
        'physician_intensivist': 'Not an intensivist (0)',
        'physician_internist': 'Not an internist (0)',
        'suspected_polytrauma': 'No suspected polytrauma (0)',
        'extremities_involved': 'No extremities involvement (0)',
        'thorax_involved': 'No thorax involvement (0)',
        'abdomen_pelvis_involved': 'No abdomen/pelvis involvement (0)',
        'spine_involved': 'No spine involvement (0)',
        'face_neck_involved': 'No face/neck involvement (0)',
        'tbi_involved': 'No traumatic brain injury (0)',
        'no_venous_access': 'Venous access present (0)',
        'night_mission': 'Day mission (0)',
        'winter_season': 'Summer season (0)',
        'winch_extraction': 'No winch extraction (0)',
    }
    if var_name in binary_reference:
        return binary_reference[var_name]

    # Fallback
    return 'Binary: 0 (reference) vs 1'

def _build_reference_table(predictor_vars):
    rows = []
    for var in predictor_vars:
        rows.append({
            'Variable': var,
            'Reference category (univariate & multivariate)': _reference_category(var)
        })
    return pd.DataFrame(rows)

print("Reference categories for adult analyses (univariate & multivariate)")
adult_reference_table = _build_reference_table(adult_predictor_vars)
display(adult_reference_table)


## Summary Figures

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Figure 3: Forest Plot - 
def create_adult_forest_plot():
    """
    Create a forest plot showing odds ratios for adult population only
    """
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Variable labels for better presentation
    var_labels = {
              'age': 'Patient age (years)',
        'dr_male_pt_male': 'Male physician - Male patient',
        'dr_male_pt_female': 'Male physician - Female patient', 
        'dr_female_pt_male': 'Female physician - Male patient',
        'physician_age': 'Physician age (years)',
        'physician_experience_years': 'Physician experience (years)',
        'physician_anesthesiologist': 'Physician anesthesiologist',
        'physician_intensivist': 'Physician intensivist',
        'physician_internist': 'Physician internist',
        'n_regions_involved': 'Number of body regions involved',
        'suspected_polytrauma': 'Suspected polytrauma',
        'extremities_involved': 'Extremities involved',
        'thorax_involved': 'Thorax involved',
        'abdomen_pelvis_involved': 'Abdomen/pelvis involved',
        'spine_involved': 'Spine involved',
        'face_neck_involved': 'Face/neck involved',
        'tbi_involved': 'Traumatic brain injury',
        'mission_duration': 'Mission duration (minutes)',
        'NACA': 'NACA score',
        'no_analgesic': 'No analgesic given',
        'opiate_ketamine_combination_therapy': 'Opioid + ketamine combination',
        'primary_mission': 'Primary mission',
        'night_mission': 'Night mission',
        'winter_season': 'Winter season',
        'winch_extraction': 'Winch extraction',
        'vas_scene': 'NRS on scene',
        'venous_access_count': 'Number of venous accesses',
        'no_venous_access': 'No venous access'
    }
    
    if len(adult_multivariate_results) > 0:
        # Prepare data from adult multivariate results
        variables = []
        ors = []
        ci_lower = []
        ci_upper = []
        pvals = []
        
        # Sort by OR magnitude for better visualization
        adult_results_sorted = adult_multivariate_results.sort_values('OR', ascending=True)
        
        for _, row in adult_results_sorted.iterrows():
            var = row['Variable']
            variables.append(var_labels.get(var, var))
            ors.append(row['OR'])
            ci_lower.append(row['CI_lower'])
            ci_upper.append(row['CI_upper'])
            pvals.append(row['P_value'])
        
        y_positions = np.arange(len(variables))
        
        # Color points based on significance
        colors = ['#FF4444' if p < 0.05 else '#666666' for p in pvals]
        marker_sizes = [120 if p < 0.05 else 80 for p in pvals]
        
        # Plot the forest plot
        for i, (or_val, ci_l, ci_u, p_val) in enumerate(zip(ors, ci_lower, ci_upper, pvals)):
            color = '#FF4444' if p_val < 0.05 else '#666666'
            size = 120 if p_val < 0.05 else 80
            
            # Plot point estimate
            ax.scatter(or_val, y_positions[i], color=color, s=size, zorder=3, alpha=0.8)
            
            # Plot confidence interval
            ax.plot([ci_l, ci_u], [y_positions[i], y_positions[i]], 
                   color=color, linewidth=2, alpha=0.7, zorder=2)
            
            # Plot CI caps
            cap_size = 0.1
            ax.plot([ci_l, ci_l], [y_positions[i]-cap_size, y_positions[i]+cap_size], 
                   color=color, linewidth=2, alpha=0.7, zorder=2)
            ax.plot([ci_u, ci_u], [y_positions[i]-cap_size, y_positions[i]+cap_size], 
                   color=color, linewidth=2, alpha=0.7, zorder=2)
            
            # Add OR and CI text
            # ax.text(max(ors) * 1.3, y_positions[i], 
            ax.text(12, y_positions[i], 
                   f'{or_val:.2f} ({ci_l:.2f}-{ci_u:.2f})', 
                   va='center', fontsize=9, fontweight='bold' if p_val < 0.05 else 'normal')
        
        # Reference line at OR = 1
        ax.axvline(x=1, color='black', linestyle='--', linewidth=1, alpha=0.8)
        
        # Formatting
        ax.set_xlabel('Odds Ratio (95% CI)', fontsize=14, fontweight='bold')
        ax.set_yticks(y_positions)
        ax.set_yticklabels(variables, fontsize=11)
        
        # Set x-axis limits with some padding
        x_min = min(min(ci_lower) * 0.8, 0.1)
        x_max = max(max(ci_upper), max(ors)) * 1.3
        ax.set_xlim(x_min, x_max)
        
        # Add grid
        ax.grid(True, alpha=0.3, axis='x')
        
        # Title and subtitle
        # ax.set_title('Factors Associated with persisting untreated pain', 
        #             fontsize=16, fontweight='bold', pad=20)
        
        # Add legend
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#FF4444', markersize=10, 
                   label=f'Significant (p < 0.05) [n={sum(1 for p in pvals if p < 0.05)}]'),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#666666', markersize=8, 
                   label=f'Not significant [n={sum(1 for p in pvals if p >= 0.05)}]')
        ]
        # ax.legend(handles=legend_elements, bbox_to_anchor=(1.05, 0), loc='lower left', fontsize=10)
        
        # Add model info text
        model_info = f"""
        Model: n = {len(adult_df_clean)} adult patients
        Outcome: Persisting untreated pain (VAS arrival > 3 & no analgesia)
        """
        # ax.text(1.02, 0.98, model_info.strip(), transform=ax.transAxes, 
        #        fontsize=9, verticalalignment='top', 
        #        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.7))
        
        plt.tight_layout()
        plt.show()
        
        # Print summary
        print("\\nForest Plot Summary - Adult Population:")
        print("=" * 50)
        print(f"Total variables analyzed: {len(variables)}")
        print(f"Significant associations (p < 0.05): {sum(1 for p in pvals if p < 0.05)}")
        print(f"Sample size: {len(adult_df_clean)} adult patients")
        print(f"Outcome rate: {adult_df_clean['persisting_untreated_pain'].mean():.1%}")
        
        # List significant variables
        sig_vars = [(var, or_val, p_val) for var, or_val, p_val in zip(variables, ors, pvals) if p_val < 0.05]
        if sig_vars:
            print("\\nSignificant associations:")
            for var, or_val, p_val in sig_vars:
                direction = "increased" if or_val > 1 else "decreased"
                print(f"  • {var}: OR = {or_val:.2f} ({direction} odds, p = {p_val:.3f})")
        else:
            print("\\nNo variables reached statistical significance (p < 0.05)")
    else:
        print("No adult multivariate results available for forest plot")

    return fig

# Create the adult-only forest plot
fig = create_adult_forest_plot()

In [ ]:
# save
# fig.savefig('/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/prehospital/analgesia/analysis/adult_trauma/untreated_persisting_pain_factors.png', dpi=300)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Forest plot using normalized mixed-effects data for adult population only

def create_adult_forest_plot_normalized():
    """
    Create a forest plot showing odds ratios for adult population only
    using normalized mixed-effects predictors (per 1 SD increase).
    """
    fig, ax = plt.subplots(figsize=(12, 10))

    # Variable labels for better presentation
    var_labels = {
        'age': 'Patient age (years)',
        'dr_male_pt_male': 'Male physician - Male patient',
        'dr_male_pt_female': 'Male physician - Female patient',
        'dr_female_pt_male': 'Female physician - Male patient',
        'physician_age': 'Physician age (years)',
        'physician_experience_years': 'Physician experience (years)',
        'physician_anesthesiologist': 'Physician anesthesiologist',
        'physician_intensivist': 'Physician intensivist',
        'physician_internist': 'Physician internist',
        'n_regions_involved': 'Number of body regions involved',
        'suspected_polytrauma': 'Suspected polytrauma',
        'extremities_involved': 'Extremities involved',
        'thorax_involved': 'Thorax involved',
        'abdomen_pelvis_involved': 'Abdomen/pelvis involved',
        'spine_involved': 'Spine involved',
        'face_neck_involved': 'Face/neck involved',
        'tbi_involved': 'Traumatic brain injury',
        'mission_duration': 'Patient-contact time (minutes)',
        'NACA': 'NACA score',
        'night_mission': 'Night mission',
        'winter_season': 'Winter season',
        'winch_extraction': 'Winch extraction',
        'vas_scene': 'NRS on scene',
        'venous_access_count': 'Number of venous accesses',
        'no_venous_access': 'No venous access'
    }

    if 'adult_mixed_fixed_df_normalized' in globals() and len(adult_mixed_fixed_df_normalized) > 0:
        # Prepare data from normalized mixed-effects fixed effects
        source_df = adult_mixed_fixed_df_normalized[
            adult_mixed_fixed_df_normalized['Variable'] != 'Intercept'
        ].copy()
        source_df = source_df.sort_values('OR', ascending=True)

        variables = []
        ors = []
        ci_lower = []
        ci_upper = []
        pvals = []

        for _, row in source_df.iterrows():
            var = row['Variable']
            variables.append(var_labels.get(var, var))
            ors.append(row['OR'])
            ci_lower.append(row['CI_lower'])
            ci_upper.append(row['CI_upper'])
            pvals.append(row['P_value'])

        y_positions = np.arange(len(variables))

        # Plot the forest plot
        for i, (or_val, ci_l, ci_u, p_val) in enumerate(zip(ors, ci_lower, ci_upper, pvals)):
            color = '#FF4444' if p_val < 0.05 else '#666666'
            size = 120 if p_val < 0.05 else 80

            # Plot point estimate
            ax.scatter(or_val, y_positions[i], color=color, s=size, zorder=3, alpha=0.8)

            # Plot confidence interval
            ax.plot([ci_l, ci_u], [y_positions[i], y_positions[i]],
                   color=color, linewidth=2, alpha=0.7, zorder=2)

            # Plot CI caps
            cap_size = 0.1
            ax.plot([ci_l, ci_l], [y_positions[i]-cap_size, y_positions[i]+cap_size],
                   color=color, linewidth=2, alpha=0.7, zorder=2)
            ax.plot([ci_u, ci_u], [y_positions[i]-cap_size, y_positions[i]+cap_size],
                   color=color, linewidth=2, alpha=0.7, zorder=2)

            # Add OR and CI text
            ax.text(max(ors) * 1.26, y_positions[i],
                   f'{or_val:.2f} ({ci_l:.2f}-{ci_u:.2f})',
                   va='center', fontsize=9, fontweight='bold' if p_val < 0.05 else 'normal')

        # Reference line at OR = 1
        ax.axvline(x=1, color='black', linestyle='--', linewidth=1, alpha=0.8)

        # Formatting
        ax.set_xlabel('Odds Ratio per 1 SD increase (95% CI)', fontsize=14, fontweight='bold')
        ax.set_yticks(y_positions)
        ax.set_yticklabels(variables, fontsize=11)

        # Set x-axis limits with some padding
        x_min = min(min(ci_lower) * 0.8, 0.1)
        x_max = max(max(ci_upper), max(ors)) * 1.3
        ax.set_xlim(x_min, x_max)

        # Add grid
        ax.grid(True, alpha=0.3, axis='x')

        # remove upper and right spine
        ax.spines[['right', 'top']].set_visible(False)

        plt.tight_layout()
        plt.show()

        # Print summary
        print("\nForest Plot Summary - Adult Population (Normalized Mixed-Effects):")
        print("=" * 62)
        print(f"Total variables analyzed: {len(variables)}")
        print(f"Significant associations (p < 0.05): {sum(1 for p in pvals if p < 0.05)}")
        print(f"Sample size: {len(adult_df_clean)} adult patients")
        print(f"Outcome rate: {adult_df_clean['persisting_untreated_pain'].mean():.1%}")

        # List significant variables
        sig_vars = [(var, or_val, p_val) for var, or_val, p_val in zip(variables, ors, pvals) if p_val < 0.05]
        if sig_vars:
            print("\nSignificant associations:")
            for var, or_val, p_val in sig_vars:
                direction = "increased" if or_val > 1 else "decreased"
                print(f"  • {var}: OR = {or_val:.2f} ({direction} odds, p = {p_val:.3f})")
        else:
            print("\nNo variables reached statistical significance (p < 0.05)")
    else:
        print("No adult normalized mixed-effects results available for forest plot")

    return fig

# Create the adult-only normalized forest plot
norm_fig = create_adult_forest_plot_normalized()

In [ ]:
# save norm fig
# norm_fig.savefig('/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/adult_trauma_analgesia/review2/mixed_effects_figures/untreated_persisting_pain_factors_mixed_normalized.png', dpi=300)

# Pediatric patients

In [ ]:
# Prepare pediatric dataset (Age < 16 years)
pediatric_df = data_df[data_df['Alter '] < 16].copy()

# insufficient pain management (VAS on arrival > 3)
pediatric_df['insufficient_pain_mgmt'] = (pediatric_df['VAS_on_arrival'] > 3).astype(int)

# Create predictor variables for pediatric patients (exclude pediatric variable)
pediatric_df['age'] = pediatric_df['Alter ']
pediatric_df['male_patient'] = (pediatric_df['Geschlecht'] == 'Männlich').astype(int)
pediatric_df['male_physician'] = (pediatric_df['physician_sex'] == 'm').astype(int)

# get only year of event
pediatric_df['event_year'] = pd.to_datetime(pediatric_df['Ereignisdatum'], format='%d.%m.%Y').dt.year
pediatric_df['physician_age'] = pediatric_df['event_year'] - pediatric_df['year_of_birth']
# physician year of final exam (from licence_date which can be either d.m.Y or Y)
pediatric_df['physician_licence_year'] = pediatric_df['licence_date'].apply(lambda x: str(x).split('.')[-1] if '.' in str(x) else str(x))
pediatric_df['physician_experience_years'] = pediatric_df['event_year'] - pd.to_numeric(pediatric_df['physician_licence_year'], errors='coerce')

pediatric_df['physician_anesthesiologist'] = pediatric_df['specialist_qualifications'].str.contains('Anaesthesiology', na=False).astype(int)
pediatric_df['physician_intensivist'] = pediatric_df['specialist_qualifications'].str.contains('Intensive care medicine', na=False).astype(int)
pediatric_df['physician_internist'] = pediatric_df['specialist_qualifications'].str.contains('General Internal Medicine|General medical practitioner', na=False).astype(int)

pediatric_df['primary_mission'] = (pediatric_df['Einsatzart'] == 'Primär').astype(int)
pediatric_df['night_mission'] = (pediatric_df['Tag oder Nacht'] == 'Nacht').astype(int)
# separate into winter half-year (October to March) and summer half-year (April to September)
pediatric_df['winter_season'] = np.where(pediatric_df['Monat'].isin(['Oktober', 'November', 'Dezember', 'Januar', 'Februar', 'März']), 1, 0).astype(int)
pediatric_df['trauma'] = pediatric_df['Einteilung (reduziert)'].str.contains('Unfall', na=False).astype(int)
pediatric_df['winch_extraction'] = pediatric_df['Bergungen'].str.contains('Winde', na=False).astype(int)
pediatric_df['vas_scene'] = pediatric_df['VAS_on_scene']
pediatric_df['mission_duration'] = (
    pd.to_datetime(pediatric_df['Übergabezeit'], format='%d.%m.%Y %H:%M:%S') - 
    pd.to_datetime(pediatric_df['Erstbefund'], format='%d.%m.%Y %H:%M:%S')
).dt.total_seconds() / 60

pediatric_df = preprocess_body_region(pediatric_df)

# number of regions involved per patient
pediatric_df['n_regions_involved'] = pediatric_df['Körperregion'].str.count(',') + 1
# if Körperregion is polytrauma, set n_regions_involved to 2
pediatric_df.loc[pediatric_df['Körperregion'].str.contains('polytrauma', na=False), 'n_regions_involved'] = 2

# regional involvement
pediatric_df['extremities_involved'] = pediatric_df['Körperregion'].str.contains('obere extremität|untere extremität', na=False).astype(int)
pediatric_df['thorax_involved'] = pediatric_df['Körperregion'].str.contains('thorax', na=False).astype(int)
# abdomen or pelvis involvement
pediatric_df['abdomen_pelvis_involved'] = pediatric_df['Körperregion'].str.contains('abdomen|becken', na=False).astype(int)
pediatric_df['spine_involved'] = pediatric_df['Körperregion'].str.contains('wirbelsäule', na=False).astype(int)
# face/neck involvement
pediatric_df['face_neck_involved'] = pediatric_df['Körperregion'].str.contains('gesicht|hals', na=False).astype(int)
# tbi 
pediatric_df['tbi_involved'] = pediatric_df['Körperregion'].str.contains('schädel-hirn', na=False).astype(int)
# polytrauma
pediatric_df['suspected_polytrauma'] = pediatric_df['Körperregion'].str.contains('polytrauma', na=False).astype(int)

# decompose patient / doctor sex into combinations
# male / male
pediatric_df['dr_male_pt_male'] = pediatric_df['male_physician'] & pediatric_df['male_patient']
pediatric_df['dr_female_pt_female'] = ((pediatric_df['male_physician'] == 0) & (pediatric_df['male_patient'] == 0)).astype(int)
pediatric_df['dr_male_pt_female'] = (pediatric_df['male_physician'] & (pediatric_df['male_patient'] == 0)).astype(int)
pediatric_df['dr_female_pt_male'] = ((pediatric_df['male_physician'] == 0) & pediatric_df['male_patient']).astype(int)

# Create the same additional variables for pediatric patients
pediatric_df['short_scene_time'] = (pediatric_df['mission_duration'] < 30).astype(int)

# Create medication dose variables (matching Table 1 approach)
pediatric_df['fentanyl_dose'] = 0
pediatric_df['ketamine_dose'] = 0
pediatric_df['esketamine_dose'] = 0
pediatric_df['morphine_dose'] = 0
pediatric_df['Alle Medikamente'] = pediatric_df['Alle Medikamente'].str.replace(',', ';')  # replace commas with semicolons for consistency
for i, row in pediatric_df.iterrows():
    if pd.isna(row['Alle Medikamente']) or row['Alle Medikamente'] == 0:
        continue
    for analgetic in row['Alle Medikamente'].split(';'):
        if analgetic.strip() == '':
            continue
        # remove mcg or mg from dose
        if '7IE' in analgetic:
                print(f"Skipping dose with 7IE: {analgetic}")
                continue

        analgetic = analgetic.replace('mcg', '').replace('mg', '').strip()
        if 'Fentanyl' in analgetic and '/h' not in analgetic:
            dose = analgetic.split('Fentanyl')[-1].strip()
            pediatric_df.at[i, 'fentanyl_dose'] += float(dose) 
        elif 'Fentanyl' in analgetic and '/h' in analgetic:
            dose = analgetic.split('Fentanyl')[-1].strip().replace('/h', '')
            dose = float(dose) * pediatric_df.at[i, 'mission_duration']  
            pediatric_df.at[i, 'fentanyl_dose'] += float(dose)
        elif 'Ketamin' in analgetic or 'Ketamine' in analgetic:
            dose = analgetic.split('Ketamin')[-1].strip()
            pediatric_df.at[i, 'ketamine_dose'] += float(dose)
        elif 'Esketamin' in analgetic:
            dose = analgetic.split('Esketamin')[-1].strip()
            pediatric_df.at[i, 'esketamine_dose'] += float(dose)
        elif 'Morphin' in analgetic or 'Morphine' in analgetic:
            dose = analgetic.split('Morphin')[-1].strip()
            pediatric_df.at[i, 'morphine_dose'] += float(dose)

# fentanyl given
pediatric_df['fentanyl_given'] = pediatric_df['fentanyl_dose'] > 0
pediatric_df['morphine_given'] = pediatric_df['morphine_dose'] > 0
pediatric_df['ketamine_given'] = pediatric_df['ketamine_dose'] > 0
pediatric_df['esketamine_given'] = pediatric_df['esketamine_dose'] > 0

# Create combined medication variables
pediatric_df['any_opiate_dose'] = pediatric_df['morphine_dose'] + pediatric_df['fentanyl_dose']
pediatric_df['any_opiate_given'] = (pediatric_df['morphine_dose'] > 0) | (pediatric_df['fentanyl_dose'] > 0)
pediatric_df['any_ketamine_given'] = (pediatric_df['ketamine_dose'] > 0) | (pediatric_df['esketamine_dose'] > 0)

# combination therapy (opiate + ketamine) vs monotherapy
pediatric_df['opiate_ketamine_combination_therapy'] = (pediatric_df['any_opiate_given'] & pediatric_df['any_ketamine_given']).astype(int)

# Create no_analgesic variable: patients who receive no analgesic
# (no_analgesic corresponds to a dose of any_opiate and any_ketamine of 0)
pediatric_df['no_analgesic'] = ((pediatric_df['any_opiate_given'] == 0) & (pediatric_df['any_ketamine_given'] == 0)).astype(int)

# create outcome variable: insufficient pain management (VAS on arrival > 3) and no analgesic given
pediatric_df['persisting_untreated_pain'] = ((pediatric_df['insufficient_pain_mgmt'] == 1) & (pediatric_df['no_analgesic'] == 1)).astype(int)

# Variables for pediatric model (including the 3 required variables)
pediatric_model_vars = ['persisting_untreated_pain', 'age', 'NACA',
                    'dr_male_pt_male', 'dr_female_pt_female', 'dr_male_pt_female', 'dr_female_pt_male',
                                        'physician_age', 'physician_experience_years', 'physician_anesthesiologist', 'physician_intensivist', 'physician_internist',
                    'n_regions_involved', 'suspected_polytrauma', 'extremities_involved', 'thorax_involved', 'abdomen_pelvis_involved',
                    'spine_involved', 'face_neck_involved', 'tbi_involved',
                   'mission_duration', 'night_mission',
                    'winch_extraction', 'vas_scene', 'winter_season']

# print number of na per variable in pediatric_model_vars
for var in pediatric_model_vars:
    n_na = pediatric_df[var].isna().sum()
    print(f"{var}: {n_na} NA values")

pediatric_df_clean = pediatric_df[pediatric_model_vars].dropna()

print(f"pediatric patients included in model: {len(pediatric_df_clean)}")



In [ ]:
# pediatric predictor variables (including the 3 required variables)
pediatric_predictor_vars = ['age', 
                        'dr_male_pt_male',  'dr_male_pt_female', 'dr_female_pt_male',
                        'physician_age', 'physician_anesthesiologist', 'physician_intensivist', 'physician_internist',
                        #   'n_regions_involved', 'extremities_involved', 'thorax_involved', 'abdomen_pelvis_involved',
                    # 'spine_involved', 'face_neck_involved', 'tbi_involved',
                        'mission_duration', 'NACA',
                        'night_mission', 'winter_season',
                       'winch_extraction', 'vas_scene']

# Rebuild cleaned dataset so physician_experience_years is fully excluded from sample selection
pediatric_required_vars = ['persisting_untreated_pain'] + list(dict.fromkeys(pediatric_predictor_vars))
pediatric_df_clean = pediatric_df[pediatric_required_vars].dropna()
print(f"Pediatric patients included in model (without physician experience): {len(pediatric_df_clean)}")

# Univariate analysis for pediatrics
pediatric_univariate_results = univariate_logistic_regression(pediatric_df_clean, 'persisting_untreated_pain', pediatric_predictor_vars)

print("pediatric PATIENTS - Univariate Analysis:")
print("=" * 60)
for _, row in pediatric_univariate_results.iterrows():
    print(f"{row['Variable']}: OR = {row['OR_CI']}, p = {row['P_formatted']}")

# Multivariate analysis for pediatrics (including the 3 required variables)
pediatric_multivariate_vars = pediatric_predictor_vars

pediatric_multivariate_results, pediatric_model_stats, pediatric_fitted_model = multivariate_logistic_regression(
    pediatric_df_clean, 'persisting_untreated_pain', pediatric_multivariate_vars)

print("\nPersisting untreated pain: (VAS on arrival > 3 and no analgesic given)")
print(f"\npediatric PATIENTS - Multivariate Analysis:")
print("=" * 60)
for _, row in pediatric_multivariate_results.iterrows():
    print(f"{row['Variable']}: OR = {row['OR_CI']}, p = {row['P_formatted']}")

print(f"\npediatric Model Statistics:")
print(f"Pseudo R-squared: {pediatric_model_stats['Pseudo R-squared']:.3f}")
print(f"AIC: {pediatric_model_stats['AIC']:.2f}")
print(f"LLR p-value: {pediatric_model_stats['LLR p-value']:.3f}")

# Multivariate analysis with normalized predictors
pediatric_multivariate_results_normalized, pediatric_model_stats_normalized, pediatric_fitted_model_normalized = multivariate_logistic_regression(
    pediatric_df_clean, 'persisting_untreated_pain', pediatric_multivariate_vars, normalize=True)

print(f"\npediatric PATIENTS - Multivariate Analysis (Normalized):")
print("=" * 60)
for _, row in pediatric_multivariate_results_normalized.iterrows():
    print(f"{row['Variable']}: OR = {row['OR_CI']}, p = {row['P_formatted']}")

# Ensure physician cluster IDs are available in cleaned frame for mixed-effects grouping
if 'physician_cluster_id' not in pediatric_df_clean.columns:
    pediatric_df_clean['physician_cluster_id'] = (
        pediatric_df.loc[pediatric_df_clean.index, 'Mitglieder mit Einsatzfunktion']
        .fillna('UNKNOWN_PHYSICIAN')
        .astype(str)
        .str.strip()
    )

# Mixed-effects fits for pediatric cohort
pediatric_mixed_result, pediatric_mixed_fixed_df = fit_mixed_effects_logit(
    pediatric_df_clean,
    outcome_var='persisting_untreated_pain',
    predictor_vars=pediatric_multivariate_vars,
    group_var='physician_cluster_id',
    label='Pediatric'
)

pediatric_mixed_result_normalized, pediatric_mixed_fixed_df_normalized = fit_mixed_effects_logit(
    pediatric_df_clean,
    outcome_var='persisting_untreated_pain',
    predictor_vars=pediatric_multivariate_vars,
    group_var='physician_cluster_id',
    label='Pediatric',
    normalize=True,
)

pediatric_multivariate_results

In [ ]:
from scipy import stats

# Pediatric mixed-effects normalized Wald and post-hoc tests for doctor/patient sex categories
sex_vars = ['dr_male_pt_male', 'dr_male_pt_female', 'dr_female_pt_male']
fixed_terms = ['Intercept'] + pediatric_multivariate_vars

# Build fixed-effects coefficient and covariance objects from normalized mixed model
beta = pd.Series(pediatric_mixed_result_normalized.fe_mean, index=fixed_terms)
p_fixed = len(fixed_terms)
cov_all = np.asarray(pediatric_mixed_result_normalized.cov_params())
cov_fixed = cov_all[:p_fixed, :p_fixed]
cov_fixed_df = pd.DataFrame(cov_fixed, index=fixed_terms, columns=fixed_terms)

# 1. Global Wald test (all sex_vars jointly): H0: beta_sex = 0
b = beta.loc[sex_vars].values.reshape(-1, 1)
V = cov_fixed_df.loc[sex_vars, sex_vars].values
W = float(b.T @ np.linalg.inv(V) @ b)
df_wald = len(sex_vars)
p_global = stats.chi2.sf(W, df_wald)

print("\nGlobal Wald test for doctor/patient sex categories (pediatric mixed-effects normalized model):")
print("=" * 100)
print(f"chi2({df_wald}) = {W:.3f}, p = {p_global:.4g}")

# 2. Pairwise post-hoc contrasts using normalized mixed-effects covariance
contrast_defs = {
    'dr_male_pt_male vs dr_male_pt_female': ('dr_male_pt_male', 'dr_male_pt_female'),
    'dr_male_pt_male vs dr_female_pt_male': ('dr_male_pt_male', 'dr_female_pt_male'),
    'dr_male_pt_female vs dr_female_pt_male': ('dr_male_pt_female', 'dr_female_pt_male'),
    'dr_male_pt_female vs dr_female_pt_female': ('dr_male_pt_female', None),
}

print("\nPairwise post-hoc contrasts for doctor/patient sex categories (pediatric mixed-effects normalized):")
print("=" * 100)

contrast_summary = []
for name, (a, b_ref) in contrast_defs.items():
    c = np.zeros(len(fixed_terms))
    c[fixed_terms.index(a)] = 1.0
    if b_ref is not None:
        c[fixed_terms.index(b_ref)] = -1.0

    est = float(c @ beta.values)
    se = float(np.sqrt(c @ cov_fixed @ c))
    z = est / se
    pval = 2 * stats.norm.sf(abs(z))

    or_val = np.exp(est)
    ci_low = np.exp(est - 1.96 * se)
    ci_high = np.exp(est + 1.96 * se)

    print(f"{name}: OR = {or_val:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {pval:.4f}")

    contrast_summary.append({
        'Contrast': name,
        'Logit_diff': est,
        'SE': se,
        'z_value': z,
        'OR': or_val,
        'CI_lower': ci_low,
        'CI_upper': ci_high,
        'p_value': pval,
    })

contrast_df = pd.DataFrame(contrast_summary)
contrast_df

In [ ]:
# Create Table 3: pediatric Patients
pediatric_table3 = create_table3(
    pediatric_univariate_results,
    pediatric_multivariate_results,
    multivariate_mixed_df=pediatric_mixed_fixed_df,
    multivariate_normalized_df=pediatric_multivariate_results_normalized,
    multivariate_mixed_normalized_df=pediatric_mixed_fixed_df_normalized,
)

print("Table 3. Factors for Persisting Untreated Pain - pediatric PATIENTS (<16 years)")
print("=" * 160)
print(f"{'Variable':<30} {'Univariate':<20} {'P-value':<10} {'Multivariate':<20} {'P-value':<10} {'Mixed Effects':<20} {'P-value':<10} {'Mixed-Normalized':<20} {'P-value':<10}")
print(f"{'':30} {'OR (95% CI)':<20} {'':10} {'OR (95% CI)':<20} {'':10} {'OR (95% CI)':<20} {'':10} {'OR (95% CI)':<20} {'':10}")
print("-" * 160)

for _, row in pediatric_table3.iterrows():
    print(
        f"{row['Variable']:<30} "
        f"{row['Univariate_OR_CI']:<20} {row['Univariate_P']:<10} "
        f"{row['Multivariate_OR_CI']:<20} {row['Multivariate_P']:<10} "
        f"{row['Multivariate_Mixed_OR_CI']:<20} {row['Multivariate_Mixed_P']:<10} "
        f"{row['Multivariate_Mixed_Normalized_OR_CI']:<20} {row['Multivariate_Mixed_Normalized_P']:<10}"
    )

print("\nOR = Odds Ratio, CI = Confidence Interval")
print("Mixed Effects = Logistic mixed-effects model with random intercept by physician")
print("Mixed-Normalized OR represents change per 1 standard deviation increase in predictor")
print(f"Model includes {len(pediatric_df_clean)} pediatric patients")
print(f"Persisting untreated pain: {pediatric_df_clean['persisting_untreated_pain'].sum()}/{len(pediatric_df_clean)} ({pediatric_df_clean['persisting_untreated_pain'].mean():.1%})")

# Show significant variables for pediatrics under normalized mixed-effects inference
pediatric_significant_vars_mixed_norm = pediatric_mixed_fixed_df_normalized[
    (pediatric_mixed_fixed_df_normalized['Variable'] != 'Intercept') & (pediatric_mixed_fixed_df_normalized['P_value'] < 0.05)
]
if len(pediatric_significant_vars_mixed_norm) > 0:
    print("\nSignificant Variables in pediatric Normalized Mixed-Effects Model (p < 0.05):")
    for _, row in pediatric_significant_vars_mixed_norm.iterrows():
        direction = "increased" if row['OR'] > 1 else "decreased"
        print(f"- {row['Variable']}: OR = {row['OR']:.2f}, {direction} odds of persisting untreated pain")
else:
    print("\nNo variables reached statistical significance (p < 0.05) in pediatric normalized mixed-effects model")

pediatric_table3

In [ ]:
# pediatric_table3.to_csv('/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/prehospital/analgesia/analysis/pediatric_trauma/table3.csv', index=False)

## Pediatric Summary figure

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Figure 3: Forest Plot - 
def create_pediatric_forest_plot():
    """
    Create a forest plot showing odds ratios for pediatric population only
    """
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Variable labels for better presentation
    var_labels = {
        'age': 'Age (years)',
        'dr_male_pt_male': 'Male doctor - Male patient',
        'dr_male_pt_female': 'Male doctor - Female patient', 
        'dr_female_pt_male': 'Female doctor - Male patient',
        'physician_age': 'Physician age (years)',
        'physician_experience_years': 'Physician experience (years)',
        'physician_anesthesiologist': 'Physician anesthesiologist',
        'physician_intensivist': 'Physician intensivist',
        'physician_internist': 'Physician internist',
        'n_regions_involved': 'Number of regions involved',
        'suspected_polytrauma': 'Suspected polytrauma',
        'extremities_involved': 'Extremities involved',
        'thorax_involved': 'Thorax involved',
        'abdomen_pelvis_involved': 'Abdomen/pelvis involved',
        'spine_involved': 'Spine involved',
        'face_neck_involved': 'Face/neck involved',
        'tbi_involved': 'Traumatic brain injury',
        'mission_duration': 'Mission duration (minutes)',
        'NACA': 'NACA score',
        'primary_mission': 'Primary mission',
        'night_mission': 'Night mission',
        'winter_season': 'Winter season',
        'winch_extraction': 'Winch extraction',
        'vas_scene': 'NRS on scene'
    }
    
    if len(pediatric_multivariate_results) > 0:
        # Prepare data from pediatric multivariate results
        variables = []
        ors = []
        ci_lower = []
        ci_upper = []
        pvals = []
        
        # Sort by OR magnitude for better visualization
        pediatric_results_sorted = pediatric_multivariate_results.sort_values('OR', ascending=True)
        
        for _, row in pediatric_results_sorted.iterrows():
            var = row['Variable']
            variables.append(var_labels.get(var, var))
            ors.append(row['OR'])
            ci_lower.append(row['CI_lower'])
            ci_upper.append(row['CI_upper'])
            pvals.append(row['P_value'])
        
        y_positions = np.arange(len(variables))
        
        # Color points based on significance
        colors = ['#FF4444' if p < 0.05 else '#666666' for p in pvals]
        marker_sizes = [120 if p < 0.05 else 80 for p in pvals]
        
        # Plot the forest plot
        for i, (or_val, ci_l, ci_u, p_val) in enumerate(zip(ors, ci_lower, ci_upper, pvals)):
            color = '#FF4444' if p_val < 0.05 else '#666666'
            size = 120 if p_val < 0.05 else 80
            
            # Plot point estimate
            ax.scatter(or_val, y_positions[i], color=color, s=size, zorder=3, alpha=0.8)
            
            # Plot confidence interval
            ax.plot([ci_l, ci_u], [y_positions[i], y_positions[i]], 
                   color=color, linewidth=2, alpha=0.7, zorder=2)
            
            # Plot CI caps
            cap_size = 0.1
            ax.plot([ci_l, ci_l], [y_positions[i]-cap_size, y_positions[i]+cap_size], 
                   color=color, linewidth=2, alpha=0.7, zorder=2)
            ax.plot([ci_u, ci_u], [y_positions[i]-cap_size, y_positions[i]+cap_size], 
                   color=color, linewidth=2, alpha=0.7, zorder=2)
            
            # Add OR and CI text
            # ax.text(max(ors) * 1.3, y_positions[i], 
            ax.text(13, y_positions[i], 
                   f'{or_val:.2f} ({ci_l:.2f}-{ci_u:.2f})', 
                   va='center', fontsize=9, fontweight='bold' if p_val < 0.05 else 'normal')
        
        # Reference line at OR = 1
        ax.axvline(x=1, color='black', linestyle='--', linewidth=1, alpha=0.8)
        
        # Formatting
        ax.set_xlabel('Odds Ratio (95% CI)', fontsize=14, fontweight='bold')
        ax.set_yticks(y_positions)
        ax.set_yticklabels(variables, fontsize=11)
        
        # Set x-axis limits with some padding
        x_min = min(min(ci_lower) * 0.8, 0.1)
        x_max = max(max(ci_upper), max(ors)) * 1.3
        ax.set_xlim(x_min, x_max)
        
        # Add grid
        ax.grid(True, alpha=0.3, axis='x')
        
        # Title and subtitle
        ax.set_title('Factors Associated with persisting untreated pain', 
                    fontsize=16, fontweight='bold', pad=20)
        
        # Add legend
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#FF4444', markersize=10, 
                   label=f'Significant (p < 0.05) [n={sum(1 for p in pvals if p < 0.05)}]'),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#666666', markersize=8, 
                   label=f'Not significant [n={sum(1 for p in pvals if p >= 0.05)}]')
        ]
        # ax.legend(handles=legend_elements, bbox_to_anchor=(1.05, 0), loc='lower left', fontsize=10)
        
        # Add model info text
        model_info = f"""
        Model: n = {len(pediatric_df_clean)} pediatric patients
        Outcome: Persisting untreated pain (VAS arrival > 3 & no analgesia)
        """
        # ax.text(1.02, 0.98, model_info.strip(), transform=ax.transAxes, 
        #        fontsize=9, verticalalignment='top', 
        #        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.7))
        
        plt.tight_layout()
        plt.show()
        
        # Print summary
        print("\\nForest Plot Summary - pediatric Population:")
        print("=" * 50)
        print(f"Total variables analyzed: {len(variables)}")
        print(f"Significant associations (p < 0.05): {sum(1 for p in pvals if p < 0.05)}")
        print(f"Sample size: {len(pediatric_df_clean)} pediatric patients")
        print(f"Outcome rate: {pediatric_df_clean['persisting_untreated_pain'].mean():.1%}")
        
        # List significant variables
        sig_vars = [(var, or_val, p_val) for var, or_val, p_val in zip(variables, ors, pvals) if p_val < 0.05]
        if sig_vars:
            print("\\nSignificant associations:")
            for var, or_val, p_val in sig_vars:
                direction = "increased" if or_val > 1 else "decreased"
                print(f"  • {var}: OR = {or_val:.2f} ({direction} odds, p = {p_val:.3f})")
        else:
            print("\\nNo variables reached statistical significance (p < 0.05)")
    else:
        print("No pediatric multivariate results available for forest plot")

    return fig

# Create the pediatric-only forest plot
ped_fig = create_pediatric_forest_plot()

In [ ]:
# ped_fig.savefig('/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/prehospital/analgesia/analysis/pediatric_trauma/insuff_pain_factors_pediatric.png', dpi=300)